In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.smpickle import load_pickle

import comp_utils

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pickle as pkl


In [5]:

# Merge the ncep pes files
ncep_pes1 = xr.open_dataset("data/ncep_gfs_pes10_20200920_20231027.nc")
ncep_pes2 = xr.open_dataset("data/ncep_gfs_pes10_20231027_20240108.nc")

ncep_pes1_df = ncep_pes1.to_dataframe()
ncep_pes2_df = ncep_pes2.to_dataframe()
ncep_pes1_df

CloudCover  SolarDownwardRadiation  \
ref_datetime valid_datetime point                                       
2020-09-20   0              0           0.000                     0.0   
                            1           0.000                     0.0   
                            2           0.000                     0.0   
                            3           0.000                     0.0   
                            4           0.000                     0.0   
...                                       ...                     ...   
2023-10-27   384            15          1.000                     0.0   
                            16          1.000                     0.0   
                            17          1.000                     0.0   
                            18          0.050                     0.0   
                            19          0.994                     0.0   

                                   Temperature   latitude  longitude  
ref_datetime valid_datetime point                                     
2020-09-20   0              0        12.650024  52.487256   0.401245  
                            1        15.150024  52.877668   0.790653  
                            2        13.050018  52.135428  -0.264034  
                            3        13.150024  52.488050  -0.126705  
                            4        14.450012  51.956370   0.658817  
...                                        ...        ...        ...  
2023-10-27   384            15        6.133728  52.558237   0.713749  
                            16        5.923706  52.447892   0.120487  
                            17        6.793732  52.521486   1.570682  
                            18        9.163727  52.877668   1.191654  
                            19        5.303711  52.078072  -0.011349  

[18918680 rows x 5 columns]

In [ ]:

# Standardize column names in hornsea2_df to match hornsea1_df
ncep_pes2_df.index.rename(['ref_datetime','valid_datetime','point'], inplace=True)

# Concatenate the DataFrames
combined_necp_df = pd.concat([ncep_pes1_df,ncep_pes2_df])


In [4]:
combined_necp_df

CloudCover  SolarDownwardRadiation  \
ref_datetime valid_datetime point                                       
2020-09-20   0              0           0.000                     0.0   
                            1           0.000                     0.0   
                            2           0.000                     0.0   
                            3           0.000                     0.0   
                            4           0.000                     0.0   
...                                       ...                     ...   
2024-01-08   384            15          0.050                     0.0   
                            16          0.050                     0.0   
                            17          0.038                     0.0   
                            18          0.052                     0.0   
                            19          0.028                     0.0   

                                   Temperature   latitude  longitude  
ref_datetime valid_datetime point                                     
2020-09-20   0              0        12.650024  52.487256   0.401245  
                            1        15.150024  52.877668   0.790653  
                            2        13.050018  52.135428  -0.264034  
                            3        13.150024  52.488050  -0.126705  
                            4        14.450012  51.956370   0.658817  
...                                        ...        ...        ...  
2024-01-08   384            15       -1.043335  52.558237   0.713749  
                            16       -1.073334  52.447892   0.120487  
                            17       -0.723328  52.521486   1.570682  
                            18        3.296661  52.877668   1.191654  
                            19       -1.763336  52.078072  -0.011349  

[20143420 rows x 5 columns]

In [ ]:

# Group by 'ref_datetime' and 'valid_datetime' and calculate the mean
combined_necp_features = combined_necp_df.groupby(['ref_datetime', 'valid_datetime']).mean()

# Reset index and then localize 'ref_datetime' to UTC and adjust 'valid_datetime'
combined_necp_features.reset_index(inplace=True)
combined_necp_features["ref_datetime"] = combined_necp_features["ref_datetime"].dt.tz_localize("UTC")
combined_necp_features["valid_datetime"] = combined_necp_features["ref_datetime"] + pd.TimedeltaIndex(combined_necp_features["valid_datetime"], unit="hours")

# Remove duplicates and keep the last occurrence
unique_necp_features = combined_necp_features.groupby('valid_datetime').last().reset_index()

# Set 'valid_datetime' as the index
unique_necp_features.set_index('valid_datetime', inplace=True)

# Merge the dwd pes files
dwd_pes1 = xr.open_dataset("data/dwd_icon_eu_pes10_20200920_20231027.nc")
dwd_pes2 = xr.open_dataset("data/dwd_icon_eu_pes10_20231027_20240108.nc")

dwd_pes1_df = dwd_pes1.to_dataframe()
dwd_pes2_df = dwd_pes2.to_dataframe()

# Standardize column names in hornsea2_df to match hornsea1_df
dwd_pes2_df.index.rename(['valid_datetime','ref_datetime','point'], inplace=True)

# Concatenate the DataFrames
combined_dwd_df = pd.concat([dwd_pes1_df,dwd_pes2_df])


